# Tuning Results Analysis

Analyze `results.tsv` from `subtitle-gen tune` runs.
Inspired by [Karpathy's autoresearch `analysis.ipynb`](https://github.com/karpathy/autoresearch) pattern.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

# Load results
tsv_path = Path('results.tsv')
df = pd.read_csv(tsv_path, sep='\t')
df['iteration'] = df['iteration'].astype(int)
print(f'Loaded {len(df)} iterations from {tsv_path}')
df.head(10)

## Composite Score Over Iterations

In [ ]:
fig, ax = plt.subplots()

colors = df['status'].map({'keep': '#3fb950', 'discard': '#f85149'})
ax.bar(df['iteration'], df['composite'], color=colors, alpha=0.8, edgecolor='white', linewidth=0.5)

# Running best line
best_so_far = df['composite'].cummax()
ax.step(df['iteration'], best_so_far, where='mid', color='#58a6ff', linewidth=2, label='Best so far')

ax.set_xlabel('Iteration')
ax.set_ylabel('Composite Score')
ax.set_title('Tuning Progress — Composite Score per Iteration')
ax.legend()
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Annotate kept iterations
for _, row in df[df['status'] == 'keep'].iterrows():
    ax.annotate(f"{row['param']}\n{row['old_value']}→{row['new_value']}",
                xy=(row['iteration'], row['composite']),
                xytext=(0, 12), textcoords='offset points',
                ha='center', fontsize=8, color='#3fb950',
                arrowprops=dict(arrowstyle='->', color='#3fb950', lw=0.8))

plt.tight_layout()
plt.show()

## Quality vs Separation Scatter

In [ ]:
fig, ax = plt.subplots()

kept = df[df['status'] == 'keep']
discarded = df[df['status'] == 'discard']

ax.scatter(discarded['quality'], discarded['separation'], c='#f85149', alpha=0.7,
           s=80, label='Discarded', edgecolors='white', linewidth=0.5, zorder=2)
ax.scatter(kept['quality'], kept['separation'], c='#3fb950', alpha=0.9,
           s=120, marker='*', label='Kept', edgecolors='white', linewidth=0.5, zorder=3)

# Label kept points
for _, row in kept.iterrows():
    ax.annotate(f"iter {int(row['iteration'])}\n{row['param']}",
                xy=(row['quality'], row['separation']),
                xytext=(8, 8), textcoords='offset points',
                fontsize=8, color='#3fb950')

# Label discarded with iteration number
for _, row in discarded.iterrows():
    ax.annotate(str(int(row['iteration'])),
                xy=(row['quality'], row['separation']),
                xytext=(5, 5), textcoords='offset points',
                fontsize=7, color='#8b949e')

# Iso-composite lines (50/50 weight)
for c in np.arange(0.4, 0.8, 0.05):
    q = np.linspace(0, 1, 100)
    s = (c - 0.5 * q) / 0.5
    mask = (s >= 0) & (s <= 1)
    ax.plot(q[mask], s[mask], '--', color='#484f58', alpha=0.3, linewidth=0.8)
    # Label at right edge
    valid_q = q[mask]
    valid_s = s[mask]
    if len(valid_q) > 0:
        ax.text(valid_q[-1], valid_s[-1], f'{c:.2f}', fontsize=7, color='#484f58', alpha=0.6)

ax.set_xlabel('Quality Score')
ax.set_ylabel('Tone Separation Score')
ax.set_title('Quality vs Separation — Pareto Frontier')
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## Parameter Exploration Heatmap

In [ ]:
# Which parameters were tried, and how did they do?
param_stats = df.groupby('param').agg(
    attempts=('iteration', 'count'),
    kept=('status', lambda x: (x == 'keep').sum()),
    best_composite=('composite', 'max'),
    avg_composite=('composite', 'mean'),
).sort_values('best_composite', ascending=False)

param_stats['keep_rate'] = (param_stats['kept'] / param_stats['attempts'] * 100).round(0).astype(int)
param_stats.style.format({'best_composite': '{:.4f}', 'avg_composite': '{:.4f}'}).bar(
    subset=['best_composite'], color='#3fb950')

## Parameter Value Trajectory

For parameters that were kept, show the trajectory of value changes.

In [ ]:
kept_params = df[df['status'] == 'keep']['param'].unique()
if len(kept_params) > 0:
    fig, axes = plt.subplots(1, len(kept_params), figsize=(6 * len(kept_params), 4), squeeze=False)
    for ax, param in zip(axes[0], kept_params):
        param_df = df[df['param'] == param].sort_values('iteration')
        # Build value trajectory: old → new for each attempt
        for _, row in param_df.iterrows():
            color = '#3fb950' if row['status'] == 'keep' else '#f85149'
            ax.annotate('', xy=(row['iteration'], row['new_value']),
                       xytext=(row['iteration'], row['old_value']),
                       arrowprops=dict(arrowstyle='->', color=color, lw=2))
            ax.scatter([row['iteration']], [row['old_value']], c='#8b949e', s=30, zorder=3)
            ax.scatter([row['iteration']], [row['new_value']], c=color, s=50, zorder=3,
                      marker='*' if row['status'] == 'keep' else 'x')
        ax.set_title(param, fontsize=11)
        ax.set_xlabel('Iteration')
        ax.set_ylabel('Value')
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    plt.tight_layout()
    plt.show()
else:
    print('No parameters were kept in this run.')

## LLM Proposer Reasoning

What reasoning did the proposer give for each attempt?

In [ ]:
for _, row in df.iterrows():
    status_icon = '✅' if row['status'] == 'keep' else '❌'
    print(f"\n{'='*80}")
    print(f"{status_icon} Iteration {int(row['iteration'])}: {row['param']} {row['old_value']} → {row['new_value']}")
    print(f"   Quality: {row['quality']:.4f}  Separation: {row['separation']:.4f}  Composite: {row['composite']:.4f}")
    print(f"   {row['description'][:200]}..." if len(str(row['description'])) > 200 else f"   {row['description']}")

## Summary Statistics

In [ ]:
print(f"Total iterations: {len(df)}")
print(f"Kept: {(df['status'] == 'keep').sum()} ({(df['status'] == 'keep').mean()*100:.0f}%)")
print(f"Discarded: {(df['status'] == 'discard').sum()} ({(df['status'] == 'discard').mean()*100:.0f}%)")
print()
print(f"Best composite: {df['composite'].max():.4f} (iter {df.loc[df['composite'].idxmax(), 'iteration']})")
print(f"Worst composite: {df['composite'].min():.4f} (iter {df.loc[df['composite'].idxmin(), 'iteration']})")
print(f"Mean composite: {df['composite'].mean():.4f}")
print()
print(f"Quality range: {df['quality'].min():.4f} – {df['quality'].max():.4f}")
print(f"Separation range: {df['separation'].min():.4f} – {df['separation'].max():.4f}")
print()
print('Unique params tried:', df['param'].nunique())
print('Params:', ', '.join(df['param'].unique()))